In [2]:
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import argparse
import pandas as pd
import json

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from esm.models.esmfold2 import (
    DNAInput,
    ESMFold2InputBuilder,
    ProteinInput,
    StructurePredictionInput,
)
from transformers.models.esmfold2.modeling_esmfold2 import ESMFold2Model

model = ESMFold2Model.from_pretrained("biohub/ESMFold2").to("cuda:2").eval()


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [3]:
def get_pep_seq(
        pdb_code:str,
        complex_base_dir:str)-> tuple:
    """从complex的pdb文件中提取pro和pep的序列"""
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_code, f'{complex_base_dir}/{pdb_code}.pdb')
    for model in structure:
        for chain in model:
            if chain.id == 'L':
                pep_sequence = ''
                for residue in chain:
                    if residue.id[0] == ' ':
                        one_letter_resname = seq1(residue.get_resname())
                        pep_sequence += one_letter_resname
            else:
                pro_sequence = ''
                for residue in chain:
                    if residue.id[0] == ' ':
                        one_letter_resname = seq1(residue.get_resname())
                        pro_sequence += one_letter_resname
    return pro_sequence, pep_sequence


for pdb in sorted(os.listdir('./test')):
    SEQ_PRO, SEQ_PEP = get_pep_seq(pdb[:-4], './test')

    spi = StructurePredictionInput(
        sequences=[
            ProteinInput(id="A", sequence=SEQ_PRO),
            ProteinInput(id="B", sequence=SEQ_PEP),
        ]
    )

    result = ESMFold2InputBuilder().fold(
        model, spi, num_loops=3, num_sampling_steps=50, num_diffusion_samples=1, seed=0
    )

    print(f"pLDDT mean: {float(result.plddt.mean()):.3f}, pTM: {float(result.ptm):.3f}, ipTM: {float(result.iptm):.3f}")

    with open(f"{pdb[:-4]}_pred.cif", "w") as f:
        f.write(result.complex.to_mmcif())


Loading CCD dictionary from /home/junjiechen/.cache/huggingface/hub/models--biohub--ESMFold2/snapshots/e1e189d0f5fb70c2693da2332eca4443c0ccccd6/ccd.pkl


[12:15:26] Depickling from a version number (16.3)that is higher than our version (16.2).
This probably won't work.
[12:15:26] Depickling from a version number (16.3)that is higher than our version (16.2).
This probably won't work.
[12:15:26] Depickling from a version number (16.3)that is higher than our version (16.2).
This probably won't work.
[12:15:26] Depickling from a version number (16.3)that is higher than our version (16.2).
This probably won't work.
[12:15:26] Depickling from a version number (16.3)that is higher than our version (16.2).
This probably won't work.
[12:15:26] Depickling from a version number (16.3)that is higher than our version (16.2).
This probably won't work.
[12:15:26] Depickling from a version number (16.3)that is higher than our version (16.2).
This probably won't work.
[12:15:26] Depickling from a version number (16.3)that is higher than our version (16.2).
This probably won't work.
[12:15:26] Depickling from a version number (16.3)that is higher than ou

pLDDT mean: 0.743, pTM: 0.654, ipTM: 0.385
